# 02 Baselines: M7 and M8

Scores the two previous methods on every held-out station. M7 is a rule and needs no fitting. M8 is trained once per distinct training set (nine fits: one per Beta fold on the other seven Beta stations, and one on all eight Beta stations shared by the ten Alpha folds) and then predicts its held-out station.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the counterfactual bridge method of the `pynrpf` package. **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes, and a *slot* counts intervals from midnight (slot 24 is 06:00).

**Inputs.** `results/01_data_folds/` and the datasets.

**Outputs.** `results/02_baselines/`: `bundles/<fold_id>/bundle.pkl` (not committed; regenerated here), one bundle manifest `<fold_id>.json` per fold, `training_summary.csv`, `intervals_m7.parquet`, `intervals_m8.parquet` (one row per quarter-hour of every held-out site-day: the method's flag and, for M8, its probabilities); `results/manifests/02_baselines.json`.

**Runtime.** About twenty minutes for the nine fits on a laptop; seconds when the bundles already exist, because a fold whose bundle is on disk is skipped unless `FORCE` is set.

**Steps.**

1. Setup.
2. Train M8 per fold and predict M7 and M8.
3. Read the training summary and the prediction tables.

## 1. Setup

Locate the article folder, import the paper code and load the configuration. Loading the configuration verifies the SHA-256 of every dataset, so a wrong or edited data file stops the run here. `CONFIG` is the one knob: point it at another YAML to run a variant into another folder.

`ONLY_FOLD` trains a single fold (for example `beta_beta_A`) to check the machinery; `FORCE` retrains folds whose bundles exist.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """publication/2_journal_article, found from this folder, JupyterLab's root or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "paper" / "stages.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "paper" / "stages.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
from paper import config, results, stages  # noqa: E402

CONFIG = ARTICLE / "config" / "evaluation.yaml"   # point this at another configuration to run a variant
SETTINGS = config.load(CONFIG)                     # verifies the dataset hashes before anything runs
RESULTS = SETTINGS.output_root()
print("results folder:", RESULTS.relative_to(ARTICLE))

ONLY_FOLD = None   # e.g. 'beta_beta_A' to train one fold only
FORCE = False      # retrain even when a bundle exists

## 2. Train and predict

M8's two classifiers are fitted with the settings of `config/evaluation.yaml` (fit window October 2023 to July 2024, validation August to September 2024, thresholds carried from the conference paper; nothing is tuned here). The held-out station is asserted absent from every training frame.

In [ ]:
out = stages.baselines(SETTINGS, fold_id=ONLY_FOLD, force=FORCE)

## 3. What was written

The training summary lists every fold's bundle, its training rows and error days, the in-bundle validation metrics and which fold it shares a bundle with. The interval tables are the common schema every method is scored through in notebook 04.

In [ ]:
training = pd.read_csv(RESULTS / '02_baselines' / 'training_summary.csv')
display(training[[c for c in training.columns if c != 'training_stations']].head(18))
for method in ('m7', 'm8'):
    table = results.intervals(SETTINGS, method)
    print(method, f'{len(table):,} interval rows,', int(table['pred_interval'].sum()), 'flagged')

## Result

M7 and M8 predictions for every held-out site-day are on disk. The next notebook scores M9 on the same days.